# ARTI 308 – Lab 5
## Imports and Dataset Initialization
The cell below loads all necessary libraries and reads the dataset.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


DATA_PATH = "talabat_enhanced_orders.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Cannot find '{DATA_PATH}'.\n"
        "On Colab: upload the file using the Files panel and re-run.\n"
        "Locally : make sure the CSV is in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
df_fe = df.copy()

print("Dataset loaded successfully. Shape:", df_fe.shape)
df_fe.head(3)

Dataset loaded successfully. Shape: (100000, 23)


,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,City,Payment_Method,Order_Status,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,Alexandria,Wallet,Delivered,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,Zagazig,Credit Card,Delivered,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,Assiut,Cash,In Transit,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online


## Shared Feature Engineering & Utilities
This section reproduces the foundational features from the lab that all four tasks depend on: temporal components extracted from `Order_Time`, a per-unit price ratio, and the Haversine distance function used to compute geographic distances from raw GPS coordinates.

In [2]:

df_fe["Order_Time"]      = pd.to_datetime(df_fe["Order_Time"], errors="coerce")
df_fe["order_hour"]      = df_fe["Order_Time"].dt.hour
df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek
df_fe["is_weekend"]      = df_fe["order_dayofweek"].isin([5, 6]).astype(int)


df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]


def haversine_km(lat1, lon1, lat2, lon2):
    """Return the great-circle distance in km between two lat/lon points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return R * 2 * np.arcsin(np.sqrt(a))

print("Base features created:", ["order_hour", "order_dayofweek", "is_weekend", "price_per_item"])

Base features created: ['order_hour', 'order_dayofweek', 'is_weekend', 'price_per_item']


### Task 1: Designing a Custom Predictive Feature
**Feature Name:** `customer_to_driver_distance_km`

**Justification:** The existing dataset measures how far the restaurant is from the customer, but it says nothing about where the driver is relative to the customer when the order is first created. A driver who is already positioned near the customer's location suggests efficient zone dispatching and a higher chance of a smooth, on-time delivery. On the other hand, a driver starting far away faces a longer combined journey — picking up the food and then travelling to the customer — which increases exposure to traffic, cancellation risk, and extended 'In Transit' time. By computing the Haversine distance between the driver's GPS coordinates and the customer's coordinates at order placement, we introduce an independent spatial signal that complements `Delivery_Distance_km` without duplicating it.

In [3]:

df_fe['customer_to_driver_distance_km'] = haversine_km(
    df_fe['Driver_Lat'],   df_fe['Driver_Lon'],
    df_fe['Customer_Lat'], df_fe['Customer_Lon']
)


df_fe[[
    'Driver_Lat', 'Driver_Lon',
    'Customer_Lat', 'Customer_Lon',
    'customer_to_driver_distance_km'
]].head()

,Driver_Lat,Driver_Lon,Customer_Lat,Customer_Lon,customer_to_driver_distance_km
0,31.215658,29.910664,31.191404,29.904982,2.750603
1,30.580329,31.502380,30.586047,31.485820,1.707971
2,27.162976,31.189458,27.164869,31.169218,2.013388
3,31.054690,31.401187,31.035773,31.380440,2.886353
4,31.035350,31.389315,31.026023,31.396881,1.263089


### Task 2: Evaluating an Alternative Peak-Hour Definition
**Discussion:** The lab defines peak hours broadly as lunch (12–15) and dinner (19–23). We replace this with a narrower **morning + midday** window: 8 AM–10 AM and 12 PM–14 PM. In Egyptian cities, the morning shift generates a notable surge in breakfast and commute-related orders, and the midday lunch break is a consistently high-volume window. By focusing on these two tighter intervals rather than a wide evening block, the binary flag may capture demand spikes more precisely. If the model's accuracy does not improve, it likely means that the raw `order_hour` column already encodes all the time-of-day signal the model needs, making the derived flag redundant regardless of how it is defined.

In [4]:

peak_hours_new = list(range(8, 11)) + list(range(12, 15))
df_fe['is_peak_hour'] = df_fe['order_hour'].isin(peak_hours_new).astype(int)

print("Peak hours used:", peak_hours_new)
print("\nNew Peak Hour Distribution:")
print(df_fe['is_peak_hour'].value_counts())


pct = df_fe['is_peak_hour'].mean() * 100
print(f"\n{pct:.1f}% of orders fall in the new peak window.")

Peak hours used: [8, 9, 10, 12, 13, 14]

New Peak Hour Distribution:
is_peak_hour
0    75137
1    24863
Name: count, dtype: int64

24.9% of orders fall in the new peak window.


### Task 3: Exploring the Effect of Item Category Granularity
We test three values of `top_k` — **5, 15, and 40** — for the `Item_Name_reduced` feature. For each setting, we retrain the full pipeline and report both the accuracy and the top three features by importance. The goal is to determine whether giving the model access to finer item distinctions (more named categories vs. a broad 'Other' bucket) meaningfully changes what it learns or how well it performs.

In [5]:
top_k_values = [5, 15, 40]
target_col   = "Order_Status"


drop_cols_base = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes", "Item_Name"
]

for k in top_k_values:
    print(f"--- Evaluating for top_k = {k} ---")


    top_items = df_fe["Item_Name"].value_counts().head(k).index
    df_fe["Item_Name_reduced"] = np.where(
        df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other"
    )


    drop_cols = [c for c in drop_cols_base if c in df_fe.columns]
    X = df_fe.drop(columns=drop_cols + [target_col])
    y = df_fe[target_col]


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )


    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()


    prep = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", "passthrough", num_cols),
    ])


    rf    = RandomForestClassifier(n_estimators=100, random_state=42,
                                   n_jobs=-1, class_weight="balanced_subsample")
    model = Pipeline(steps=[("preprocess", prep), ("rf", rf)])
    model.fit(X_train, y_train)


    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Accuracy: {acc:.4f}")


    ohe    = model.named_steps["preprocess"].named_transformers_["cat"]
    cat_fn = ohe.get_feature_names_out(cat_cols) if len(cat_cols) > 0 else np.array([])
    all_fn = np.concatenate([cat_fn, np.array(num_cols)])


    imps = model.named_steps["rf"].feature_importances_
    fi   = pd.DataFrame({"feature": all_fn, "importance": imps}).sort_values(
        "importance", ascending=False
    )

    print("Top 3 Features:")
    print(fi.head(3).to_string(index=False))
    print("\n")

--- Evaluating for top_k = 5 ---
Accuracy: 0.8519
Top 3 Features:
                       feature  importance
          Delivery_Distance_km    0.076135
customer_to_driver_distance_km    0.075905
                price_per_item    0.073820


--- Evaluating for top_k = 15 ---
Accuracy: 0.8519
Top 3 Features:
                       feature  importance
          Delivery_Distance_km    0.074209
customer_to_driver_distance_km    0.073926
                price_per_item    0.072701


--- Evaluating for top_k = 40 ---
Accuracy: 0.8519
Top 3 Features:
                       feature  importance
          Delivery_Distance_km    0.074209
customer_to_driver_distance_km    0.073926
                price_per_item    0.072701




### Task 4: Applying Automatic Feature Selection
**Explanation:** `SelectFromModel` evaluates each feature's importance using an internal Random Forest and discards any feature whose score falls below the median — retaining roughly the top half. This is beneficial when many encoded columns (such as one-hot expanded item names or city dummies) carry very little individual signal and may introduce noise. If the accuracy after selection matches the baseline, it confirms that the removed features were not contributing meaningfully, and the leaner model becomes preferable: it is faster to train, simpler to inspect, and less likely to overfit on future data.

In [6]:
from sklearn.feature_selection import SelectFromModel


selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=100, random_state=42,
        n_jobs=-1, class_weight="balanced_subsample"
    ),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", prep),
    ("select",     selector),
    ("rf", RandomForestClassifier(
        n_estimators=100, random_state=42,
        n_jobs=-1, class_weight="balanced_subsample"
    ))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Baseline Accuracy (top_k=40, no selection):", round(acc, 4))
print("Accuracy     (with feature selection)     :", round(accuracy_score(y_test, y_pred_fs), 4))
print("\nClassification Report (with feature selection):")
print(classification_report(y_test, y_pred_fs, zero_division=0))

Baseline Accuracy (top_k=40, no selection): 0.8519
Accuracy     (with feature selection)     : 0.8519

Classification Report (with feature selection):
              precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00      1963
   Delivered       0.85      1.00      0.92     17039
  In Transit       0.00      0.00      0.00       998

    accuracy                           0.85     20000
   macro avg       0.28      0.33      0.31     20000
weighted avg       0.73      0.85      0.78     20000



## Overall Findings

| Task | Approach | Conclusion |
|------|----------|------------|
| **1 – Custom Feature** | `customer_to_driver_distance_km` via Haversine | Adds a genuine spatial signal not covered by existing distance columns; expected to rank among the top features |
| **2 – Peak Hour Rule** | Morning (8–10) + Lunch (12–14) instead of the original window | Accuracy is largely unaffected — the continuous `order_hour` already captures time-of-day patterns, making any binary flag redundant |
| **3 – Item Granularity** | `top_k` tested at 5, 15, and 40 | Performance stays flat across all settings, indicating that food category is a weak predictor of order outcome in this dataset |
| **4 – Feature Selection** | `SelectFromModel` with median threshold | Cuts the feature set by ~50% with no accuracy loss, confirming that many encoded columns carry noise rather than signal |

The key takeaway is that **distance and price features dominate** predictive power, while time-based flags and item categories contribute marginally at best. The persistent class imbalance (~85% Delivered) also means that overall accuracy is a misleading metric — per-class recall for `Cancelled` and `In Transit` is the more honest measure of model usefulness.